<a href="https://colab.research.google.com/github/samaaturky/ML/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

* **The selected lane**: Refresh / Content Opportunity Scoring (Lane 2). I'm choosing this because notebook 02 already showed a measurable gap between a hand-written rule and a learned model on the same precision@K metric — the rule wins at the very top of the list, the model wins deeper into it — which means there's real, non-trivial signal here worth formalizing into a proper ranked queue. I may pivot toward Lane 4 if CTR-gap analysis turns out to be the stronger thread; I have until end of Week 4 to confirm.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** out of thousands of content pages, which ones should an editor review
for refresh FIRST, given limited review capacity?

**Unit of analysis:** one page (`content_id`), aggregated over a trailing 90-day
window (starter data) or a future warehouse-built window (capstone)

**Output:** a ranked queue of pages, each with a score and a reason code.

**Action:** an editor opens the top N pages and decides: refresh, expand, protect,prune, or leave alone.

**Cost of a wrong call:** a false positive wastes an editor's limited time on a page
that wasn't actually declining. A false negative lets a genuinely declining,
high-traffic page keep losing visibility unnoticed. Because review capacity is
scarce, precision@K matters more here than overall accuracy.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os, sys, subprocess
# setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

In [3]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns\n")

30,000 rows, 44 columns



In [4]:
# Number 1: does the obvious signal even work?
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"1. search_volume vs impressions_90d correlation: {corr:.3f}")
print("near zero: keyword search volume alone is NOT a usable priority signal.\n")

1. search_volume vs impressions_90d correlation: 0.001
near zero: keyword search volume alone is NOT a usable priority signal.



In [5]:
# Number 2: is there a real rule-vs-model gap to exploit? (from notebook 02's results)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = (df["trend_direction"].str.lower() == "down").astype(int).values
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]

p20 = precision_at_k(hand_rule_score, y, 20)
p50 = precision_at_k(hand_rule_score, y, 50)
print(f"2. Hand rule Precision@20: {p20:.3f}  |  Precision@50: {p50:.3f}")
print(" strong at the top, weaker deeper in the list -> room for a model to add value.\n")

2. Hand rule Precision@20: 0.900  |  Precision@50: 0.680
 strong at the top, weaker deeper in the list -> room for a model to add value.



In [7]:
# Number 3: how common is the (proxy) positive label -> is ranking even meaningful here?
base_rate = y.mean()
print(f"3. Declining rate (trend_direction == 'down'): {base_rate:.3f}")
print(" just over half of pages carry this label -> enough positives to rank,")
print(" but this base rate is the floor any model/rule must beat.")

3. Declining rate (trend_direction == 'down'): 0.542
 just over half of pages carry this label -> enough positives to rank,
 but this base rate is the floor any model/rule must beat.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

* This is cross-sectional, observational data — one snapshot, no experiment. I can say
a page's signals are **associated with** decline or opportunity, and that a ranked
queue is **decision-support**. I CANNOT claim that refreshing a page will **cause**
recovery — that requires a causal design (an A/B test or similar) I am not running.

* The starter label (`is_declining_label = trend_direction == "down"`) is a proxy
computed from the CURRENT window, not a future outcome. Notebook 02 showed exactly
why this matters: feeding `trend_pct` (the number the label is derived from) into a
model as a feature gave Precision@50 = 1.000 — leakage, not skill. My capstone target
should move toward a genuine forward-looking window (features from a prior period
predicting an outcome in a later period) once I move to the warehouse release.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.